# Ingest circuit.cvs file
- Read the file using spark dataframe reader api
- Add metadata columns
    - Source File
    - Ingest timestamp
- Write to bronze delta table

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
    spark.read
    .format('csv')
    .option('header', 'true')
    # .option('inferSchema', 'true')
    .option('mode', 'FAILFAST')
    .schema(circuits_schema)
    .load(source_file))

In [0]:
display(circuits_df)

In [0]:
circuits_final_df = add_ingestions_metadata(circuits_df)
                    
                                

In [0]:
display(circuits_final_df)

Write to bronze delta table

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table('formula1.bronze.circuits'))